# 02b — Seed pipeline with hand-curated Material/

Reads `Train/Material/curated.jsonl` (the hand-curated, `aro check`-
validated examples written by `Train/tools/curate_material.py`) and any
per-prompt training pairs from `Train/Material/<slug>.json` (produced
by `Train/tools/run_prompts.py`), and folds them into the shared
`PAIRS_FILE` via `save_notebook_pair` so EVERY downstream stage picks
them up:

  * NB04 (warm-start fine-tune) — model gets correct ARO from day one
  * NB17 (dataset assembly) → NB18 (SFT)
  * NB19 (preference SFT) — used as `chosen` for matching prompts
  * NB21 (iterative loop) — included in cumulative dataset
  * NB23 (distillation) — teacher trains on the same gold set

NB26 (the Material-focused fine-tune) is the final booster on top.
This notebook ensures the same data is also present *throughout* the
pipeline so every model touch-point benefits.

In [ ]:
import sys
from pathlib import Path
_cfg_dir = Path('.').resolve()
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import importlib, config
importlib.reload(config)
from config import *

import json
from collections import Counter

MATERIAL_DIR = Path(SCRIPT_DIR).parent / 'Material'
CURATED_FILE = MATERIAL_DIR / 'curated.jsonl'

print(f'Material dir: {MATERIAL_DIR}')
print(f'  curated.jsonl exists: {CURATED_FILE.exists()}')
print(f'  per-prompt JSONs:     {len([p for p in MATERIAL_DIR.glob("*.json") if p.name != "canonical.json"])}')

In [ ]:
# Drop any pairs left over from a previous run so a re-execution doesn't
# duplicate them in PAIRS_FILE.
clean_notebook_pair('02b_material')
clean_notebook_pair('02b_material_runner')

In [ ]:
# 1. Hand-curated examples — already pre-validated by curate.py.
curated_pairs = []
if CURATED_FILE.exists():
    with open(CURATED_FILE) as f:
        for line in f:
            if not line.strip():
                continue
            ex = json.loads(line)
            instr  = ex.get('instruction', '').strip()
            output = ex.get('output', '').strip()
            if not instr or not output:
                continue
            curated_pairs.append({
                'instruction': instr,
                'output':      output,
                'task_type':   ex.get('task_type', 'code_generation'),
                'source':      f'curated/{ex.get("category", "misc")}',
                'score':       1.0,
            })

saved = save_notebook_pairs('02b_material', curated_pairs)
print(f'Saved {saved} curated pairs to {PAIRS_FILE}')

In [ ]:
# 2. Per-prompt training pairs from run_prompts.py. Only those with a
#    non-null `expected` field — those are the ones we have ground
#    truth for.
runner_pairs = []
for p in sorted(MATERIAL_DIR.glob('*.json')):
    if p.name == 'canonical.json':
        continue
    try:
        rec = json.loads(p.read_text())
    except json.JSONDecodeError:
        continue
    instr    = (rec.get('prompt') or '').strip()
    expected = (rec.get('expected') or '').strip()
    if not instr or not expected:
        continue
    runner_pairs.append({
        'instruction': instr,
        'output':      expected,
        'task_type':   'code_generation',
        'source':      f'runner/{p.stem}',
        'score':       1.0,
    })

saved = save_notebook_pairs('02b_material_runner', runner_pairs)
print(f'Saved {saved} runner-derived pairs to {PAIRS_FILE}')

In [ ]:
totals = Counter()
totals['curated'] = len(curated_pairs)
totals['runner']  = len(runner_pairs)
totals['total']   = sum(totals.values())
for name, n in totals.most_common():
    print(f'  {name:8} {n:>5}')
if totals['total'] == 0:
    print()
    print('No Material pairs found. Run:')
    print('  python3 Train/tools/curate_material.py')
    print('  python3 Train/tools/run_prompts.py')
    print('and re-execute this notebook.')